# A1组 6.2 习题参考解答代码（修改版）

> 配套：本目录 `A1组-习题-6.2.md`、`A1组-习题参考解答-6.2.md`、`A1-G12-正文-6.2.md`。

- 从原 `A1组-习题参考解答代码-6.2-6.3.ipynb` 拆出 6.2-1、6.2-2 两个单元；6.3 部分见 `A1组-习题参考解答代码-6.3.ipynb`。
- 相对路径：从本修改版目录根启动 Jupyter；图写入 `图片/`，结果 JSON 写入 `数据/`。
- 环境：Python 3.12.x（原始运行 3.12.15）+ NumPy 2.5.4 + PyTorch CPU + Matplotlib。

### 运行说明

本 Notebook 的两个代码单元（含多组训练）使用固定种子；在本机 8 核 CPU 上完整串行运行约 10–30 分钟（小张量、float64 的逐点优化占主要时间；本批保存输出的计时合计约 307 s）。为避免 Notebook 携带超长日志，训练输出取自**同一代码的原始运行结果**，并已保存为 `数据/A1组-6.2-习题1结果.json` 等文件；附录残差检查在本环境实际执行。代码已加入 `torch.set_num_threads(1)`，用于降低小张量在线程间同步的开销，不改变算法。

> 修改版说明：本 Notebook 的输出为原运行的保存输出，本轮未重训；所有 Adam 已显式写出 `betas=(0.9, 0.999), eps=1e-8`，与 PyTorch 默认一致，不改变数值。


## 题 6.2-1 参数缩放对称与观测等价

对应题面 (1)–(4)。本单元完成：解析解与边界通量；固定 $(k,r)$ 的两组 PINN 正问题；从两种初值出发的反演；损失景观扫描。重点结论：$(k,r)=(1,4)$ 与 $(0.25,1)$ 给出同一条解，仅靠内部解观测无法区分；边界通量 $q(0),q(1)=-ku'$ 可以区分。

输出：`数据/A1组-6.2-习题1结果.json`；图见 `图片/A1组-6.2-习题1*.png`。

In [1]:
import os, json, math, time
import numpy as np
import torch
torch.set_num_threads(1)
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float64)
np.random.seed(0)
torch.manual_seed(0)

ROOT = "数据"
FIG = "图片"
assert os.path.isdir(FIG), "缺少 图片/ 目录；请在提交包根目录下启动 Jupyter"

m_true = 2.0
kA, rA = 1.0, 4.0
kB, rB = 0.25, 1.0

def u_exact(x, k, r):
    m = math.sqrt(r/k)
    if abs(m) < 1e-12:
        return x
    return np.sinh(m*x)/np.sinh(m)

x_obs = (np.arange(21)+0.5)/21.0
u_obs = u_exact(x_obs, kA, rA)
sigma = 0.005

def q0(k, r):
    m = math.sqrt(r/k)
    return -k*m/math.sinh(m)

def q1(k, r):
    m = math.sqrt(r/k)
    return -k*m/math.tanh(m)

class MLP(nn.Module):
    def __init__(self, width=64, depth=3):
        super().__init__()
        layers = [nn.Linear(1, width), nn.Tanh()]
        for _ in range(depth-1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, 1)]
        self.net = nn.Sequential(*layers)
        for p in self.parameters():
            if p.ndim == 2:
                nn.init.xavier_normal_(p)
    def forward(self, x):
        return self.net(x)

def u_ansatz(model, x):
    # hard constraint u(0)=0, u(1)=1
    return x + x*(1.0-x)*model(x)

def pde_residual(model, x, k, r):
    x = x.clone().requires_grad_(True)
    u = u_ansatz(model, x)
    u_x = torch.autograd.grad(u.sum(), x, create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x.sum(), x, create_graph=True)[0]
    return -k*u_xx + r*u

def train_fixed(k, r, seed=0, iters=8000):
    torch.manual_seed(seed)
    model = MLP()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.999), eps=1e-8)  # 显式写出默认超参
    x_obs_t = torch.tensor(x_obs.reshape(-1,1))
    u_obs_t = torch.tensor(u_obs.reshape(-1,1))
    x_col = torch.linspace(0,1,400).reshape(-1,1)
    x_bc = torch.tensor([[0.0],[1.0]])
    hist = []
    for it in range(iters):
        opt.zero_grad()
        u_pred = u_ansatz(model, x_obs_t)
        loss_d = torch.mean((u_pred-u_obs_t)**2)
        res = pde_residual(model, x_col, k, r)
        loss_p = torch.mean(res**2)
        u_bc = u_ansatz(model, x_bc)
        loss_b = torch.mean((u_bc - torch.tensor([[0.0],[1.0]]))**2)
        loss = loss_d + 1e-2*loss_p + 1e2*loss_b
        loss.backward()
        opt.step()
        if it % 100 == 0:
            hist.append([it, float(loss_d), float(loss_p), float(loss_b)])
    xg = np.linspace(0,1,201)
    with torch.no_grad():
        u_pred = u_ansatz(model, torch.tensor(xg.reshape(-1,1))).numpy().ravel()
    ue = u_exact(xg, k, r)
    rel = np.linalg.norm(u_pred-ue)/np.linalg.norm(ue)
    # data metrics
    with torch.no_grad():
        u_at_obs = u_ansatz(model, torch.tensor(x_obs.reshape(-1,1))).numpy().ravel()
        bc = u_ansatz(model, torch.tensor([[0.0],[1.0]])).numpy().ravel()
    res = pde_residual(model, torch.linspace(0,1,240).reshape(-1,1), k, r).detach().numpy().ravel()
    return dict(model=model, hist=hist, xg=xg, u_pred=u_pred, ue=ue, rel=float(rel),
                data_rmse=float(np.sqrt(np.mean((u_at_obs-u_obs)**2))),
                pde_rms=float(np.sqrt(np.mean(res**2))),
                bc_max=float(np.max(np.abs(bc-np.array([0.0,1.0])))))

def train_inverse(init_k, init_r, seed=0, iters=8000):
    torch.manual_seed(seed)
    model = MLP()
    logk = nn.Parameter(torch.tensor(math.log(init_k)))
    logr = nn.Parameter(torch.tensor(math.log(init_r)))
    opt = torch.optim.Adam(list(model.parameters())+[logk, logr], lr=2e-3, betas=(0.9,0.999), eps=1e-8)
    x_obs_t = torch.tensor(x_obs.reshape(-1,1))
    u_obs_t = torch.tensor(u_obs.reshape(-1,1))
    x_col = torch.linspace(0,1,400).reshape(-1,1)
    x_bc = torch.tensor([[0.0],[1.0]])
    hist = []
    for it in range(iters):
        opt.zero_grad()
        k = torch.exp(logk); r = torch.exp(logr)
        u_pred = u_ansatz(model, x_obs_t)
        loss_d = torch.mean((u_pred-u_obs_t)**2)
        res = pde_residual(model, x_col, k, r)
        loss_p = torch.mean(res**2)
        u_bc = u_ansatz(model, x_bc)
        loss_b = torch.mean((u_bc - torch.tensor([[0.0],[1.0]]))**2)
        loss = loss_d + 1e-2*loss_p + 1e2*loss_b
        loss.backward()
        opt.step()
        if it % 100 == 0:
            hist.append([it, float(loss_d), float(loss_p), float(loss_b), float(torch.exp(logk)), float(torch.exp(logr))])
    kk, rr = float(torch.exp(logk)), float(torch.exp(logr))
    with torch.no_grad():
        u_at_obs = u_ansatz(model, torch.tensor(x_obs.reshape(-1,1))).numpy().ravel()
    return dict(k=kk, r=rr, q0=q0(kk,rr), data_rmse=float(np.sqrt(np.mean((u_at_obs-u_obs)**2))), hist=hist)

t0 = time.time()
fixed = {}
for name,(k,r) in {"A":(kA,rA), "B":(kB,rB)}.items():
    fixed[name] = train_fixed(k,r,seed=0,iters=8000)
    print(name, fixed[name]["rel"], fixed[name]["data_rmse"], fixed[name]["pde_rms"], fixed[name]["bc_max"])

inv1 = train_inverse(1.0,4.0,seed=1,iters=8000)
inv2 = train_inverse(0.5,1.0,seed=2,iters=8000)
print("inv1", inv1["k"], inv1["r"], inv1["q0"], inv1["data_rmse"])
print("inv2", inv2["k"], inv2["r"], inv2["q0"], inv2["data_rmse"])

# loss landscape from analytic u
kg = np.linspace(0.1, 2.0, 161)
rg = np.linspace(0.2, 8.0, 161)
K, R = np.meshgrid(kg, rg)
L = np.zeros_like(K)
for i in range(K.shape[0]):
    for j in range(K.shape[1]):
        uu = u_exact(x_obs, K[i,j], R[i,j])
        L[i,j] = np.mean((uu-u_obs)**2)

# figures
xg = fixed["A"]["xg"]
plt.figure(figsize=(6.4,4.2))
plt.plot(xg, fixed["A"]["ue"], 'k-', lw=2.2, label='analytic A/B')
plt.plot(xg, fixed["A"]["u_pred"], 'C0--', lw=1.6, label='PINN A')
plt.plot(xg, fixed["B"]["u_pred"], 'C1:', lw=1.6, label='PINN B')
plt.xlabel('x'); plt.ylabel('u(x)'); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG,"A1组-6.2-习题1双参数场.png"), dpi=180); plt.close()

plt.figure(figsize=(6.4,4.6))
cs = plt.contourf(K, R, np.log10(L+1e-30), levels=40, cmap='viridis')
plt.colorbar(cs, label=r'$\log_{10}$ data MSE')
plt.plot([kA,kB],[rA,rB],'o', color='red', ms=7, label='A and B')
plt.plot(kg, 4*kg, 'w--', lw=1.2, label=r'$r=4k$')
plt.xlabel('k'); plt.ylabel('r'); plt.legend(loc='upper left'); plt.tight_layout()
plt.savefig(os.path.join(FIG,"A1组-6.2-习题1损失山谷.png"), dpi=180); plt.close()

plt.figure(figsize=(6.0,4.0))
xx = np.arange(2); w=0.35
plt.bar(xx-w/2, [q0(kA,rA), q1(kA,rA)], w, label='A (1,4)')
plt.bar(xx+w/2, [q0(kB,rB), q1(kB,rB)], w, label='B (0.25,1)')
plt.xticks(xx, ['q(0)','q(1)']); plt.ylabel('flux'); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG,"A1组-6.2-习题1边界通量.png"), dpi=180); plt.close()

plt.figure(figsize=(6.4,4.0))
for name in ['A','B']:
    h = np.array(fixed[name]["hist"])
    plt.semilogy(h[:,0], h[:,1]+1e-16, label=f'data {name}')
    plt.semilogy(h[:,0], h[:,2]+1e-16, '--', label=f'PDE {name}')
plt.xlabel('iteration'); plt.ylabel('loss'); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG,"A1组-6.2-习题1训练曲线.png"), dpi=180); plt.close()

res = {
  "analytic": {"max_abs_uA_uB": float(np.max(np.abs(u_exact(xg,kA,rA)-u_exact(xg,kB,rB)))),
               "q0_A": q0(kA,rA), "q0_B": q0(kB,rB), "q1_A": q1(kA,rA), "q1_B": q1(kB,rB),
               "q0_diff": q0(kA,rA)-q0(kB,rB), "q1_diff": q1(kA,rA)-q1(kB,rB)},
  "fixed": {name: {kk: fixed[name][kk] for kk in ["rel","data_rmse","pde_rms","bc_max"]} for name in fixed},
  "inverse": {"init_A": {kk: inv1[kk] for kk in ["k","r","q0","data_rmse"]},
               "init_B": {kk: inv2[kk] for kk in ["k","r","q0","data_rmse"]},
               "q0_A_true": q0(kA,rA), "q0_B_true": q0(kB,rB)},
  "time_sec": time.time()-t0,
  "loss_min": float(L.min()), "loss_max": float(L.max())
}
with open(os.path.join(ROOT,"A1组-6.2-习题1结果.json"),"w",encoding="utf-8") as f:
    json.dump(res,f,ensure_ascii=False,indent=2)
print(json.dumps(res,ensure_ascii=False,indent=2))



{
  "analytic": {
    "max_abs_uA_uB": 0.0,
    "q0_A": -0.5514411295435665,
    "q0_B": -0.13786028238589162,
    "q1_A": -2.0746294414550963,
    "q1_B": -0.5186573603637741,
    "q0_diff": -0.41358084715767485,
    "q1_diff": -1.5559720810913222
  },
  "fixed": {
    "A": {
      "rel": 1.6070779042220406e-05,
      "data_rmse": 7.602229316705281e-06,
      "pde_rms": 0.0014882022081310457,
      "bc_max": 0.0
    },
    "B": {
      "rel": 3.0962858297149044e-05,
      "data_rmse": 1.464758917867401e-05,
      "pde_rms": 0.0006155801817453438,
      "bc_max": 0.0
    }
  },
  "inverse": {
    "init_A": {
      "k": 0.9163447396150999,
      "r": 3.665306941914612,
      "q0": -0.5053155128815252,
      "data_rmse": 9.348695870114462e-06
    },
    "init_B": {
      "k": 0.35520620153922755,
      "r": 1.4208002107396203,
      "q0": -0.19587713089869624,
      "data_rmse": 7.947577845386675e-06
    },
    "q0_A_true": -0.5514411295435665,
    "q0_B_true": -0.13786028238589162
  },


### 6.2-1 结果图

![6.2-1 双参数场](图片/A1组-6.2-习题1双参数场.png)

![6.2-1 损失山谷](图片/A1组-6.2-习题1损失山谷.png)

![6.2-1 边界通量](图片/A1组-6.2-习题1边界通量.png)

![6.2-1 训练曲线](图片/A1组-6.2-习题1训练曲线.png)

## 题 6.2-2 正问题与反问题的损失及观测算子

对应题面 (1)–(4)。本单元完成：正问题 PINN；反问题只用内部数据估计 $a$；仅边界点布局的 $\chi^2(a)$ 平坦性；FIM 与 20 次噪声实现的标准差。重点结论：正问题没有数据项仍可解；反问题靠数据项识别 $a$，仅边界点对 $a$ 为零信息。

输出：`数据/A1组-6.2-习题2结果.json`；图见 `图片/A1组-6.2-习题2*.png`。

In [2]:
import os, json, math, time
import numpy as np
import torch
torch.set_num_threads(1)
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
# 本机交付环境未安装 scipy；用有界黄金分割搜索实现等价的一维极小化。
import math as _math

class _OptResult:
    pass

def minimize_scalar(fun, bounds=(0.0, 1.0), method="bounded", options=None, **_):
    """有界一维极小化（黄金分割），对应 scipy.optimize.minimize_scalar(method='bounded')。"""
    a, b = float(bounds[0]), float(bounds[1])
    tol = float((options or {}).get("xatol", 1e-10))
    gr = (_math.sqrt(5.0) - 1.0) / 2.0
    c, d = b - gr * (b - a), a + gr * (b - a)
    fc, fd = fun(c), fun(d)
    for _ in range(200):
        if b - a <= tol:
            break
        if fc < fd:
            b, d, fd = d, c, fc
            c = b - gr * (b - a)
            fc = fun(c)
        else:
            a, c, fc = c, d, fd
            d = a + gr * (b - a)
            fd = fun(d)
    x = 0.5 * (a + b)
    out = _OptResult()
    out.x = x
    out.fun = float(fun(x))
    out.success = True
    return out


torch.set_default_dtype(torch.float64)
np.random.seed(1); torch.manual_seed(1)
ROOT = "数据"
FIG = "图片"
assert os.path.isdir(FIG), "缺少 图片/ 目录；请在提交包根目录下启动 Jupyter"

pi2=math.pi**2
def u_true(a,x):
    return (pi2+4)/(pi2+a)*np.sin(np.pi*x)

x_int=np.arange(1,9)/9.0
sigma=0.01
u_obs=u_true(4.0,x_int)+sigma*np.random.randn(8)

class MLP(nn.Module):
    def __init__(self,width=64,depth=3):
        super().__init__()
        layers=[nn.Linear(1,width),nn.Tanh()]
        for _ in range(depth-1):
            layers += [nn.Linear(width,width),nn.Tanh()]
        layers += [nn.Linear(width,1)]
        self.net=nn.Sequential(*layers)
        for p in self.parameters():
            if p.ndim==2: nn.init.xavier_normal_(p)
    def forward(self,x): return self.net(x)

def ansatz(model,x):
    return x*(1.0-x)*model(x)

def pde_res(model,x,a):
    x=x.clone().requires_grad_(True)
    u=ansatz(model,x)
    ux=torch.autograd.grad(u.sum(),x,create_graph=True)[0]
    uxx=torch.autograd.grad(ux.sum(),x,create_graph=True)[0]
    f=(pi2+4)*torch.sin(math.pi*x)
    return -uxx+a*u-f

def train_forward(seed=0,iters=6000):
    torch.manual_seed(seed)
    model=MLP()
    opt=torch.optim.Adam(model.parameters(),lr=1e-3,betas=(0.9,0.999),eps=1e-8)
    xcol=torch.linspace(0,1,300).reshape(-1,1)
    hist=[]
    for it in range(iters):
        opt.zero_grad()
        res=pde_res(model,xcol,4.0)
        loss=torch.mean(res**2)
        loss.backward(); opt.step()
        if it%200==0: hist.append([it,float(loss.detach())])
    xg=np.linspace(0,1,201)
    with torch.no_grad():
        up=ansatz(model,torch.tensor(xg.reshape(-1,1))).numpy().ravel()
    ue=np.sin(math.pi*xg)
    res=pde_res(model,torch.linspace(0,1,300).reshape(-1,1),4.0).detach().numpy().ravel()
    return dict(model=model,hist=hist,xg=xg,up=up,ue=ue,
                rel=float(np.linalg.norm(up-ue)/np.linalg.norm(ue)),
                pde_rms=float(np.sqrt(np.mean(res**2))),
                bc_max=float(max(abs(up[0]),abs(up[-1]))))

def train_inverse(seed=0,iters=6000):
    torch.manual_seed(seed)
    model=MLP()
    loga=nn.Parameter(torch.tensor(math.log(2.0)))
    opt=torch.optim.Adam(list(model.parameters())+[loga],lr=1e-3,betas=(0.9,0.999),eps=1e-8)
    xobs_t=torch.tensor(x_int.reshape(-1,1)); uobs_t=torch.tensor(u_obs.reshape(-1,1))
    xcol=torch.linspace(0,1,300).reshape(-1,1)
    hist=[]
    for it in range(iters):
        opt.zero_grad()
        a=torch.exp(loga)
        up=ansatz(model,xobs_t)
        loss_d=torch.mean((up-uobs_t)**2)/sigma**2
        res=pde_res(model,xcol,a)
        loss_p=torch.mean(res**2)
        loss=loss_d+1e-2*loss_p
        loss.backward(); opt.step()
        if it%200==0: hist.append([it,float(loss_d.detach()),float(loss_p.detach()),float(a.detach())])
    a=float(torch.exp(loga))
    with torch.no_grad():
        up=ansatz(model,xobs_t).numpy().ravel()
    return dict(a=a,data_rmse=float(np.sqrt(np.mean((up-u_obs)**2))),hist=hist)

t0=time.time()
fwd=train_forward(seed=0,iters=6000)
inv=train_inverse(seed=1,iters=6000)
print("forward",fwd["rel"],fwd["pde_rms"],fwd["bc_max"])
print("inverse",inv["a"],inv["data_rmse"])

# chi2 scans
a_grid=np.linspace(0.01,10,501)
chi_int=np.zeros_like(a_grid)
for i,a in enumerate(a_grid):
    chi_int[i]=np.sum((u_true(a,x_int)-u_obs)**2)/sigma**2
# boundary layout: u_a(0)=u_a(1)=0, observations zero (no noise here)
chi_bnd=np.zeros_like(a_grid)  # constant zero
# with noise on boundary it is still constant; report normalized constant
eps_b=np.array([0.0,0.0])
chi_bnd=np.full_like(a_grid, np.sum(eps_b**2)/sigma**2)

# FIM at a=4
du_da=-(pi2+4)/(pi2+4.0)**2*np.sin(np.pi*x_int)
I=float(np.sum(du_da**2)/sigma**2)
std_fim=1/math.sqrt(I)
# 4-point design
x4=np.arange(1,5)/5.0
du_da4=-(pi2+4)/(pi2+4.0)**2*np.sin(np.pi*x4)
I4=float(np.sum(du_da4**2)/sigma**2)
std4=1/math.sqrt(I4)

# 20 noise realizations, analytic least squares via scalar minimization
ests=[]; rng=np.random.default_rng(123)
for k in range(20):
    uo=u_true(4.0,x_int)+sigma*rng.standard_normal(8)
    fun=lambda a: np.sum((u_true(a,x_int)-uo)**2)/sigma**2
    r=minimize_scalar(fun,bounds=(0.01,10),method='bounded',options={'xatol':1e-10})
    ests.append(r.x)
ests=np.array(ests)

# figures
plt.figure(figsize=(6.4,4.2))
plt.plot(fwd["xg"],fwd["ue"],'k-',lw=2.2,label='analytic')
plt.plot(fwd["xg"],fwd["up"],'C0--',lw=1.6,label='PINN forward')
plt.xlabel('x'); plt.ylabel('u(x)'); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG,"A1组-6.2-习题2正问题解.png"),dpi=180); plt.close()

fig,ax=plt.subplots(1,2,figsize=(10,4))
ax[0].semilogy(a_grid,chi_int+1e-12,'C0-')
ax[0].axvline(4,color='k',ls='--',lw=1)
ax[0].set_xlabel('a'); ax[0].set_ylabel(r'$\chi^2(a)$ (interior)'); ax[0].set_title('8 interior sensors')
ax[1].plot(a_grid,chi_bnd,'C1-')
ax[1].set_xlabel('a'); ax[1].set_ylabel(r'$\chi^2(a)$ (boundary)'); ax[1].set_title('boundary sensors only')
ax[1].set_ylim(-0.1,0.1)
plt.tight_layout(); plt.savefig(os.path.join(FIG,"A1组-6.2-习题2卡方曲线.png"),dpi=180); plt.close()

plt.figure(figsize=(6.4,4.2))
xx=np.linspace(0,1,201)
for a in [2,4,6]:
    plt.plot(xx,u_true(a,xx),label=f'a={a}')
plt.plot(x_int,u_obs,'ko',ms=4,label='observations')
plt.xlabel('x'); plt.ylabel('u'); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG,"A1组-6.2-习题2不同a的解.png"),dpi=180); plt.close()

plt.figure(figsize=(6.4,4.0))
plt.plot(np.array(fwd["hist"])[:,0],np.array(fwd["hist"])[:,1]+1e-16,label='forward PDE loss')
plt.yscale('log'); plt.xlabel('iteration'); plt.ylabel('loss'); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG,"A1组-6.2-习题2训练曲线.png"),dpi=180); plt.close()

res={
 "forward":{"rel":fwd["rel"],"pde_rms":fwd["pde_rms"],"bc_max":fwd["bc_max"]},
 "inverse_pinn":{"a":inv["a"],"data_rmse":inv["data_rmse"]},
 "fim":{"I":I,"std":std_fim,"I4":I4,"std4":std4},
 "noise20":{"median_abs_err":float(np.median(np.abs(ests-4))),"mean":float(np.mean(ests)),"std":float(np.std(ests,ddof=1)),"min":float(ests.min()),"max":float(ests.max())},
 "chi2":{"a_at_min":float(a_grid[np.argmin(chi_int)]),"min":float(chi_int.min()),"bnd_variation":float(chi_bnd.max()-chi_bnd.min())},
 "time_sec":time.time()-t0
}
with open(os.path.join(ROOT,"A1组-6.2-习题2结果.json"),"w",encoding="utf-8") as f:
    json.dump(res,f,ensure_ascii=False,indent=2)
print(json.dumps(res,ensure_ascii=False,indent=2))


{
  "forward": {
    "rel": 8.926606561214939e-06,
    "pde_rms": 0.002270142186283181,
    "bc_max": 0.0
  },
  "inverse_pinn": {
    "a": 3.981700434770421,
    "data_rmse": 0.008823954614976783
  },
  "fim": {
    "I": 233.9291623986534,
    "std": 0.06538194216256712,
    "I4": 129.9606457770297,
    "std4": 0.08771908030587582
  },
  "noise20": {
    "median_abs_err": 0.045842291800404,
    "mean": 3.974255917983828,
    "std": 0.05068896973411544,
    "min": 3.8530356181111967,
    "max": 4.069620144729059
  },
  "chi2": {
    "a_at_min": 4.04596,
    "min": 13.525975088585357,
    "bnd_variation": 0.0
  },
  "time_sec": 89.37853455543518
}


### 6.2-2 结果图

![6.2-2 正问题解](图片/A1组-6.2-习题2正问题解.png)

![6.2-2 卡方曲线](图片/A1组-6.2-习题2卡方曲线.png)

![6.2-2 不同 a 的解](图片/A1组-6.2-习题2不同a的解.png)

![6.2-2 训练曲线](图片/A1组-6.2-习题2训练曲线.png)